In [14]:

# import vari
import numpy as np
import pandas as pd
import matplotlib.pyplot as pt
import seaborn as sb


In [ ]:
# 100k righe occupano ~140mb di ram, tutto il file è decisamente troppo massiccio
# sample = pd.read_csv('./data/data.csv' , nrows=100000)
# sample.info(memory_usage='deep')

samplesize = 0.05

# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ genera sample

# rng = np.random.default_rng(42)
# # quindi leggiamo il file a chunk di 100k righe 
# chunks = pd.read_csv('./data/data.csv', chunksize=100_000)
# # e per ogni chunk ne tiriamo fuori un pezzetto casuale, di grandezza un ventesimo, e concateniamo tutto
# sample = pd.concat(chunk.sample(frac=samplesize, random_state=rng) for chunk in chunks)
# # e poi salviamo in un pkl per non dover rivare tutto il giro ogni volta
# sample.to_pickle(f'./data/sample_{samplesize}.pkl')

# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ carica sample
sample = pd.read_pickle(f'./data/sample_{samplesize}.pkl')

dataset = sample



In [31]:
dataset.info()

<class 'pandas.DataFrame'>
Index: 435718 entries, 47367 to 8703763
Data columns (total 20 columns):
 #   Column                       Non-Null Count   Dtype 
---  ------                       --------------   ----- 
 0   ioCountry                    435718 non-null  str   
 1   postid                       435718 non-null  str   
 2   post_text                    435717 non-null  str   
 3   application_name             435718 non-null  str   
 4   post_language                430510 non-null  str   
 5   in_reply_to_postid           44407 non-null   str   
 6   in_reply_to_accountid        43798 non-null   str   
 7   post_time                    435718 non-null  str   
 8   accountid                    435718 non-null  str   
 9   account_profile_description  352369 non-null  str   
 10  follower_count               435643 non-null  object
 11  following_count              435665 non-null  object
 12  account_creation_date        435718 non-null  str   
 13  is_repost                

follower_count e following_count dovrebbero essere numerici, come mai sono object? Come mai lo è anche is_control?
Viene fuori che nei following/follower alcuni numeri sono scritti letterali, e in is control convivono booleani e testuali

In [ ]:
# pd.to_numeric(dataset['following_count'])
# pd.to_numeric(dataset['follower_count'])
# dataset['is_control'].unique()

array(['True', 'False', True, False, 'is_control?'], dtype=object)

In [29]:
valuesPeek = pd.DataFrame({
    'dtype': sample.dtypes,
    'n': sample.nunique(),
    'n_missing': sample.isna().sum(),
    'percent_missing': (sample.isna().sum() / len(sample) * 100),
    'values': pd.Series({c: sample[c].value_counts().head(10).index.tolist()
                     for c in sample.columns}),
})

valuesPeek

,dtype,n,n_missing,percent_missing,values
ioCountry,str,5,0,0.000000,"[russia_1, uae, ecuador, iran_1, iran_4]"
postid,str,435657,0,0.000000,"[7ef1b9781ec94d8d7f9335df62531828afb6ceb0ae, e6c07be11f97a6513b896a66fbc8bab10c605567d5, 307fb68239daebdc7ff1509951d4fc9639558b5272, be906754ae3e5c597fc4c6a49d636d2f14b748c75e, 5d85298c6ad8c92e6730825960a11087bfbba50ceb, af782aab6ccd802fae71ed9be32c5a2642c0ec258f, 3e00ad9b53b524c18633f85dc97691b3dae8c34807, 0b8fb89f74350a357f2236585684b78e973ec65ab4, 79ae0676f0e7b22dd3631bd44b5886b8ff04bfe0c4, 99adb6de529d001118815168cb33e83c8b81cfa9fd]"
post_text,str,409245,1,0.000230,"[#خادم_الحرمين_الشريفين الملك سلمان بن عبدالعزيز، في خطاب شامل بمناسبة افتتاح أعمال الدورة الجديدة لـ #مجلس_الشورى السعودي: ند…, #خادم_الحرمين_الشريفين الملك سلمان بن عبدالعزيز، في خطاب شامل بمناسبة افتتاح أعمال الدورة الجديدة لـ #مجلس_الشورى السعودي: ال…, Just setting up my Twitter. #myfirstTweet, مباشر | كلمة الشيخ ⁧#حمدان_بن_محمد_بن_راشد⁩ ولي عهد ⁧#دبي⁩ بـ ⁧#القمة_العالمية_للحكومات https://eb29d1c6e01e003aa9ea48fd5f3941fccfdb0f0568, على شعب #قطر التفكير جلياً فيما آلت اليه اوضاع الشعب القطري بسبب السياسات العدائيه التي تقوم بها الحكومة القطرية ..\n1- عزلة الش…, صناعة الثقافة في #الإمارات ونموذج التسامح كقوة ناعمة @67cfead2f185696240217e0126cfcebfcd5f559f6e https://768a6b0ee33fb79b70a321ce1f1baf9ec673a5ccc9, خالد بن ضحي @259449496fb820b1e1e1ec2aede409fde9633b34a3 يكتب مقالا بعنوان: رسائل الشيخ #عبدالله_بن_زايد.. مواجهة الأزمات كلمة السر\n\nرابـط المقال | https://1dcb70f62fb80364d546f7007195cc705f5ffee773, تحذيرات من اضطراب البحر وارتفاع الموج في #الخليج_العربي\n\nرابـط الخبر | https://8593ee0cf1d6c0692d26a8eb87003490c8968480ad https://12023e449af9ca0bbe6eeebd541b66ceb351c5e2c1, #خادم_الحرمين_الشريفين الملك سلمان بن عبدالعزيز، في خطاب شامل بمناسبة افتتاح أعمال الدورة الجديدة لـ #مجلس_الشورى السعودي: #ا…, فيديو بعدسة #فرسان_الإمارات لقبيلة #البحارنة\nخلال مشاركتها في #مسيرة_الإتحاد_للقبائل ضمن #مهرجان_الشيخ_زايد_التراثي بالوثبة…]"
application_name,str,2294,0,0.000000,"[62010408238d4a0f958ec97eec3db0a4f69bcd0d11, 15592fcc82d5064a8f4a3b89774e6cd0bbd2118aa0, 042bc4a3f4973060f4b166ba92732fc75b9f58ffc8, 072f1b013621f1b0d9753c98d97f5a0a97a8b68303, b9052708258f305d40d00a13826c6cb32520a2d48c, 7e031f14b59c758dde50006135cc62f80536b3a0cf, 9a596c0f7e3412e41e5c5c232d584467395174cc96, 69e041bbce9125f1439977ae72ba0304219aafa953, f8cdaad9d4dbe8caecfb812ba13ea76ef99009305c, 6d157c13e82961d5e70b891da588f96bd56a3d7728]"
post_language,str,56,5208,1.195268,"[en, ru, ar, es, fr, und, fa, pt, tr, ja]"
in_reply_to_postid,str,43425,391311,89.808316,"[5e4e007fca5d5c205cb2526b82d5ae003ceccf0684, bc10b4c06c56c8f6dc905f77e963c158eeee765033, a4ba43dc199867a323465fe7c2d32280ac5d08bcba, e9eff26134ca4cd5667910ec8499f7b42ff7ad4837, 658acd8353391f62a2e60b67fc37ae6639e65f674d, 7bcc609f7cb30824237a26839fb0c9d8b7c8bf0311, 40637d1c73db6f2c73685fbfc309a515c577b9ed09, af75e1bd7f7fbb05036d53954b7840b23fa4f70eb5, a260e6e15fd5dc470e44c5ed8758f82485a831ba14, 719a7c24a12f308d3ec3e3c70d9d45092a37ccbe96]"
in_reply_to_accountid,str,33019,391920,89.948086,"[bef0e7981715abed0539a1eb8eedfdbf59651779f2, aae2514dd2b49b2e9005f27a9b7c88b4368e7b0c2d, 356b94f846e87ef3ad4e46c8847006b951d69314dc, 4fe4f6286353d7fcc9d334023a4f033ce4ba1cb87d, ef79bf5d1f19d67b33fa5b89bf42ea55f9a42aeda2, 8fa3cfbe1b9cf47694243be193d6c39167f901856c, b9cc268271ebd71a94c64e2e10e2dbee4ae24557d0, 3336e292394f9e218f5819c5113f6206470be9ed8e, 5124e7aca91e6a49ca9d5d87edabd6bf0b09fc95bf, df486c86ec15ff62a3509a5e3733340c2bbcdb8913]"
post_time,str,395526,0,0.000000,"[2018-03-23 11:01:00, 2018-03-27 00:01:00, 2019-01-05 17:58:00, 2019-02-24 17:16:00, 2019-02-24 15:10:00, 2019-02-26 01:16:00, 2019-03-06 09:02:00, 2017-11-04 00:15:00, 2018-03-14 00:21:00, 2019-02-26 19:44:00]"
accountid,str,52517,0,0.000000,"[ce3438d4a69ae98420a7a3c449366da079d00df09b, a3a36ee39837796a9ce6986ce093dad383312ee95b, abd07c34899a0b8e28b8bb88222a6633d9502679de, 71cc57155d061d447c2c65756c7dfd6873e403a097, 294e449ba806914bff056a2cac0128731967

In [30]:
sample['post_language'].unique()


<StringArray>
[ 'es',  'en', 'und',   nan,  'lv',  'fr',  'it',  'in',  'pt',  'de',  'nl',
  'tl',  'ca',  'ja',  'ru',  'sv',  'tr',  'uk',  'lt',  'hi',  'ko',  'ht',
  'pl',  'vi',  'et',  'ro',  'cs',  'hu',  'eu',  'cy',  'no',  'da',  'th',
  'fi',  'zh',  'el',  'ur',  'ar',  'is',  'iw',  'sl',  'id',  'bn',  'kn',
  'sk',  'bg',  'fa',  'bs',  'si',  'ps',  'sr',  'sd',  'ne', 'ckb',  'hy',
  'ta',  'ml']
Length: 57, dtype: str